[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/15_Gas_Consumption.ipynb)

# Notebook 15 — Gas Consumption

**Companion to Chapter 15**

This laboratory models remaining gas as a resource state driven by depth and uncertain breathing demand.

> The calculations are educational and omit operational reserve and contingency rules. Do not use them to plan a dive.

## Learning objectives

By the end of this notebook, you should be able to:

- convert surface-equivalent breathing rate into depth-dependent demand;
- integrate remaining gas over a variable profile;
- compare profiles with the same maximum depth;
- interpret future gas horizons as conditional predictions;

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})
rho, g, p0 = 1025.0, 9.80665, 101325.0

def ambient_pressure_pa(depth_m):
    return p0 + rho*g*np.asarray(depth_m)

def ambient_pressure_bar(depth_m):
    return ambient_pressure_pa(depth_m)/1e5

## 1. Exact pressure ratio and the 10 m approximation

In [ ]:
z=np.linspace(0,40,401)
exact=ambient_pressure_pa(z)/p0
approx=1+z/10
plt.plot(z,exact,label="hydrostatic")
plt.plot(z,approx,"--",label="1 + z/10")
plt.xlabel("Depth [m]"); plt.ylabel("Absolute-pressure ratio")
plt.title("Pressure ratio used in gas-demand models"); plt.legend(); plt.show()
print(f"Maximum relative difference: {np.max(np.abs(exact-approx)/exact)*100:.2f}%")

## 2. Depth-dependent demand

In [ ]:
def gas_rate_lpm(depth_m,surface_rate_lpm,workload=1.0):
    return surface_rate_lpm*workload*ambient_pressure_pa(depth_m)/p0

assert np.isclose(gas_rate_lpm(0,18.0),18.0)
assert gas_rate_lpm(20,18.0) > gas_rate_lpm(10,18.0)
for depth in [0,10,20,30,40]:
    print(depth,gas_rate_lpm(depth,18.0))

## 3. Integrate a complete profile

The example profile exists only to test the state equation.

In [ ]:
def depth_profile(t_min):
    t=np.asarray(t_min)
    return np.piecewise(t,[t<3,(t>=3)&(t<18),(t>=18)&(t<21),t>=21],
        [lambda x:20*x/3,20,lambda x:20*(21-x)/3,0])

def workload(t_min):
    t=np.asarray(t_min)
    return 1+0.65*((t>=10)&(t<14))

t=np.linspace(0,25,2501); z=depth_profile(t)
rate=gas_rate_lpm(z,18,workload(t))
used=np.concatenate([[0],np.cumsum((rate[1:]+rate[:-1])*np.diff(t)/2)])
g0=2400.0; remaining=g0-used
fig,ax=plt.subplots(3,1,sharex=True,figsize=(8,8))
ax[0].plot(t,z); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
ax[1].plot(t,rate); ax[1].set_ylabel("Rate [L/min]")
ax[2].plot(t,remaining); ax[2].set(xlabel="Time [min]",ylabel="Gas [surface L]")
plt.show()
assert np.all(np.diff(remaining)<=1e-10)
assert np.min(remaining) >= 0

## 4. Same maximum depth, different history

In [ ]:
def rectangular_use(bottom_minutes):
    # idealized 3 min descent, bottom, 3 min ascent
    tt=np.linspace(0,bottom_minutes+6,int((bottom_minutes+6)*100)+1)
    zz=np.where(tt<3,20*tt/3,np.where(tt<3+bottom_minutes,20,np.maximum(0,20*(bottom_minutes+6-tt)/3)))
    return np.trapezoid(gas_rate_lpm(zz,18),tt)

for bottom in [5,10,15]:
    print(f"{bottom:2d} min bottom -> {rectangular_use(bottom):.0f} surface L")

## 5. Conditional forecast range

In [ ]:
g_now=900.0; depth_now=20.0
rates=[14,18,25]
for qs in rates:
    q=gas_rate_lpm(depth_now,qs)
    print(f"Assumed surface rate {qs:2d} L/min -> depletion horizon {g_now/q:.1f} min")

## Engineering exercises

1. Compare a shallow-long and deep-short profile with equal duration.
2. Add uncertainty bands to workload rather than one deterministic multiplier.
3. Convert remaining free gas to idealized pressure for two cylinder volumes.
4. Test the sensitivity to surface pressure and water density.


In [ ]:
# Exercise starter: compare uncertainty in surface-equivalent demand
exercise_surface_rates = np.array([14.0, 18.0, 25.0])
# Integrate each scenario over the same depth profile.


## Summary

Gas depletion is an integral of pressure-scaled respiratory demand. The full depth and workload histories matter, and every future horizon is conditional on explicit assumptions.